# RoadAnomaly — Stage 5 final test evaluation (one shot)

**Run only after** `FINAL_MODEL_SELECTION_FREEZE.json` exists.

This notebook:
1. Verifies Drive checkpoint SHA-256 against freeze metadata.
2. Verifies split lock.
3. Evaluates E0/E1/E2a/E2b **once** on the test split at frozen operational thresholds.
4. Writes results under Drive `experiments/results/phase4/` and metadata `…/phase4/`.

Does **not** retune thresholds. Does **not** retrain.

In [ ]:
from pathlib import Path
import hashlib, json, os, subprocess, sys
import pandas as pd

DRIVE_ROOT = Path("/content/drive/MyDrive/RoadAnomaly")
CLONE_DIR = Path("/content/RoadAnomaly")
GITHUB_REPO = "https://github.com/AhmedChatti/RoadAnomaly.git"
GITHUB_BRANCH = "main"
GITHUB_COMMIT_SHA = None  # use latest main after pull; freeze file is on main
GITHUB_TOKEN_SECRET_NAME = "GITHUB_TOKEN"
EXPECTED_SPLIT = "9a01c05cfc7d33783c4f64b91b0ae423a4d3ad57439fd71ffc029a8f04585c78"
AUTHORIZE_TEST_EVALUATION = True  # set True only for the approved one-shot run


In [ ]:
from google.colab import drive, userdata
drive.mount("/content/drive")

os.environ["ROADANOMALY_ARTIFACTS_ROOT"] = str(DRIVE_ROOT)
os.environ["ROADANOMALY_FRAMES_DIR"] = str(DRIVE_ROOT / "dataset" / "original" / "frames")
os.environ["ROADANOMALY_CHECKPOINTS_DIR"] = str(DRIVE_ROOT / "checkpoints")
os.environ["ROADANOMALY_METADATA_DIR"] = str(DRIVE_ROOT / "experiments" / "metadata")
os.environ["ROADANOMALY_RESULTS_DIR"] = str(DRIVE_ROOT / "experiments" / "results")
os.environ["ROADANOMALY_FIGURES_DIR"] = str(DRIVE_ROOT / "experiments" / "figures")

token = userdata.get(GITHUB_TOKEN_SECRET_NAME)
auth_url = GITHUB_REPO.replace("https://", f"https://x-access-token:{token}@")

def run(cmd, cwd=None):
    print("+", " ".join(cmd))
    subprocess.run(cmd, cwd=cwd, check=True)

if not CLONE_DIR.exists():
    run(["git", "clone", "--branch", GITHUB_BRANCH, auth_url, str(CLONE_DIR)])
else:
    run(["git", "remote", "set-url", "origin", auth_url], cwd=str(CLONE_DIR))
    run(["git", "fetch", "origin"], cwd=str(CLONE_DIR))
    run(["git", "checkout", "-B", GITHUB_BRANCH, f"origin/{GITHUB_BRANCH}"], cwd=str(CLONE_DIR))
    run(["git", "reset", "--hard", f"origin/{GITHUB_BRANCH}"], cwd=str(CLONE_DIR))
if GITHUB_COMMIT_SHA:
    run(["git", "checkout", GITHUB_COMMIT_SHA], cwd=str(CLONE_DIR))
print("HEAD", subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=str(CLONE_DIR), text=True).strip())
os.environ["ROADANOMALY_ROOT"] = str(CLONE_DIR)
sys.path.insert(0, str(CLONE_DIR / "src"))
os.chdir(CLONE_DIR)
_freeze_clone = CLONE_DIR / "experiments/metadata/phase4/FINAL_MODEL_SELECTION_FREEZE.json"
print("freeze in clone?", _freeze_clone.exists(), _freeze_clone)


In [ ]:
assert AUTHORIZE_TEST_EVALUATION, "Set AUTHORIZE_TEST_EVALUATION=True only for the frozen one-shot test run."

from dataset import load_split_manifest, verify_split_lock
from evaluation import evaluate_predictor_on_split
from models.baseline import AllBackgroundModel
from losses import bce_dice_loss
from tensorflow import keras
import shutil

def sha256_file(path: Path) -> str:
    h = hashlib.sha256()
    with path.open('rb') as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

split_sha = verify_split_lock(load_split_manifest())
assert split_sha == EXPECTED_SPLIT
print("split_ok", split_sha)

candidates = [
    Path.cwd() / "experiments/metadata/phase4/FINAL_MODEL_SELECTION_FREEZE.json",
    CLONE_DIR / "experiments/metadata/phase4/FINAL_MODEL_SELECTION_FREEZE.json",
    DRIVE_ROOT / "experiments/metadata/phase4/FINAL_MODEL_SELECTION_FREEZE.json",
]
freeze_path = next((c for c in candidates if c.is_file()), None)

EMBEDDED_FREEZE = json.loads('{"frozen_at_utc":"2026-11-08T17:45:00+00:00","purpose":"Freeze validation-based model and threshold choices before any test-set access (Stage 5).","split_manifest_sha256":"9a01c05cfc7d33783c4f64b91b0ae423a4d3ad57439fd71ffc029a8f04585c78","git_commit_sha_training":"9e734a1d0af48342f92c084216b1dd1d4c254741","selection_split":"val","selection_metric":"mean_per_image_mean_dice","primary_model":{"experiment_id":"E2b_unet_mobilenetv2_finetune_20261108_1721","reason":"Highest validation mean Dice among E0/E1/E2a/E2b","operational_threshold":0.55,"training_threshold":0.5,"checkpoint_path_drive":"/content/drive/MyDrive/RoadAnomaly/checkpoints/phase4/E2b_unet_mobilenetv2_finetune_20261108_1721/best.keras","checkpoint_sha256":"280170dc64347b0b4cc8ce3af9be40dec4ac3733b27d1abeb620cc0c8e88e432"},"models_to_evaluate_once_on_test":[{"experiment_id":"E0_all_background","operational_threshold":0.5,"normalize":"scale_0_1","notes":"trivial baseline; no checkpoint"},{"experiment_id":"E1_cnn_scratch_20261108_1510","operational_threshold":0.35,"normalize":"scale_0_1","checkpoint_path_drive":"/content/drive/MyDrive/RoadAnomaly/checkpoints/phase3/E1_cnn_scratch_20261108_1510/best.keras","checkpoint_sha256":"567bde326ee5b776bc34132b23012543b35f195ec8d60229d01196eb1a0fb234"},{"experiment_id":"E2a_unet_mobilenetv2_frozen_20261108_1714","operational_threshold":0.15,"normalize":"mobilenet_v2","checkpoint_path_drive":"/content/drive/MyDrive/RoadAnomaly/checkpoints/phase4/E2a_unet_mobilenetv2_frozen_20261108_1714/best.keras","checkpoint_sha256":"428845e976a8f4773ce929ec66520227b49928142d4c34e38974af671f38b080"},{"experiment_id":"E2b_unet_mobilenetv2_finetune_20261108_1721","operational_threshold":0.55,"normalize":"mobilenet_v2","checkpoint_path_drive":"/content/drive/MyDrive/RoadAnomaly/checkpoints/phase4/E2b_unet_mobilenetv2_finetune_20261108_1721/best.keras","checkpoint_sha256":"280170dc64347b0b4cc8ce3af9be40dec4ac3733b27d1abeb620cc0c8e88e432"}],"rules":["Do not retune thresholds on test.","Do not change model selection using test metrics.","Evaluate each listed model at most once on test under this freeze.","allow_test=True only inside the Stage 5 script/notebook."]}')

if freeze_path is None:
    print("WARN: freeze JSON not found in clone/Drive; using embedded freeze.")
    freeze = EMBEDDED_FREEZE
    freeze_path = DRIVE_ROOT / "experiments/metadata/phase4/FINAL_MODEL_SELECTION_FREEZE.json"
    freeze_path.parent.mkdir(parents=True, exist_ok=True)
    freeze_path.write_text(json.dumps(freeze, indent=2), encoding="utf-8")
    print("Wrote freeze to Drive:", freeze_path)
else:
    freeze = json.loads(freeze_path.read_text(encoding="utf-8"))
    print("Loaded freeze from", freeze_path)
    drive_freeze = DRIVE_ROOT / "experiments/metadata/phase4/FINAL_MODEL_SELECTION_FREEZE.json"
    drive_freeze.parent.mkdir(parents=True, exist_ok=True)
    if freeze_path.resolve() != drive_freeze.resolve():
        shutil.copy2(freeze_path, drive_freeze)
        print("Copied freeze to Drive:", drive_freeze)

print("primary", freeze["primary_model"]["experiment_id"])

out_dir = DRIVE_ROOT / "experiments" / "results" / "phase4"
meta_dir = DRIVE_ROOT / "experiments" / "metadata" / "phase4"
out_dir.mkdir(parents=True, exist_ok=True)
meta_dir.mkdir(parents=True, exist_ok=True)

guard = meta_dir / "STAGE5_TEST_EVAL_DONE.json"
if guard.exists():
    raise RuntimeError(f"Test evaluation already recorded at {guard}. Refusing a second run.")

rows = []
for spec in freeze["models_to_evaluate_once_on_test"]:
    eid = spec["experiment_id"]
    thr = float(spec["operational_threshold"])
    normalize = spec["normalize"]
    print("Evaluating", eid, "thr", thr)

    if eid.startswith("E0"):
        model = AllBackgroundModel()
        predict_fn = lambda x, m=model: m.predict_proba(x)
        ckpt_sha = None
        ckpt_ok = True
    else:
        ckpt = Path(spec["checkpoint_path_drive"])
        assert ckpt.is_file(), ckpt
        ckpt_sha = sha256_file(ckpt)
        expected = spec["checkpoint_sha256"]
        ckpt_ok = ckpt_sha == expected
        print("  sha", ckpt_sha, "match", ckpt_ok)
        assert ckpt_ok, (ckpt_sha, expected)
        loaded = keras.models.load_model(
            ckpt,
            custom_objects={"bce_dice": bce_dice_loss(positive_weight=15.0)},
            compile=False,
        )
        predict_fn = lambda x, m=loaded: m.predict(x, verbose=0)

    result = evaluate_predictor_on_split(
        predict_fn,
        "test",
        threshold=thr,
        normalize=normalize,
        batch_size=4,
        allow_test=True,
    )
    per = result["per_image"]
    per_path = out_dir / f"{eid}_test_per_image.csv"
    per.to_csv(per_path, index=False)
    mp = result["mean_per_image"]
    rows.append({
        "experiment_id": eid,
        "split": "test",
        "threshold": thr,
        "normalize": normalize,
        "checkpoint_sha256": ckpt_sha,
        "checkpoint_sha_match": ckpt_ok,
        **mp,
        "global_dice": result["global"]["dice"],
        "global_iou": result["global"]["iou"],
        "global_accuracy": result["global"]["accuracy"],
        "per_image_csv": str(per_path),
    })

summary = pd.DataFrame(rows)
summary_path = out_dir / "phase4_final_test_comparison.csv"
summary.to_csv(summary_path, index=False)
report = {
    "stage": 5,
    "status": "completed",
    "split": "test",
    "split_manifest_sha256": split_sha,
    "freeze_file": str(freeze_path),
    "primary_model": freeze["primary_model"]["experiment_id"],
    "summary_csv": str(summary_path),
    "rows": rows,
    "notes": "One-shot test evaluation after frozen validation selection. No threshold retuning.",
}
guard.write_text(json.dumps(report, indent=2), encoding="utf-8")
print(summary)
print("Wrote", summary_path)
print("Wrote", guard)
